# Fig S1 — Genomic oncoprint (Fig S1A–B)

Execute cells in order.

Set `DLBCL_DATA_DIR` before the setup cell to override the data root.
Example: `DLBCL_DATA_DIR = '/path/to/data'`

In [1]:
%matplotlib inline

import os
import sys
from pathlib import Path

NOTEBOOK_DIR = Path.cwd().resolve()
REPO_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dlbcl.notebook_setup import run_notebook_setup

_ctx = run_notebook_setup('discovery', 'supplemental_fig1')
REPO_ROOT = _ctx.repo_root
_paths = _ctx.paths
FIG_DIR = _ctx.fig_dir
adata = _ctx.adata
arch_df = _ctx.arch_df
pred = _ctx.pred
gep = _ctx.gep
surv = _ctx.surv
archetype_label = _ctx.archetype_label
PATIENT_SUBSET = _ctx.patient_subset
OUTDIR = FIG_DIR
OUTDIR.mkdir(parents=True, exist_ok=True)

# Override data root: export DLBCL_DATA_DIR=/path/to/data before running.
print(f"AnnData: {_paths.adata_path}")
print(f"Figures: {FIG_DIR}")

import gc

import pandas as pd
import scanpy as sc

ADATA_PATH = _paths.adata_path

import shutil
import subprocess

from dlbcl.genomic_profiling import (
    GENOMIC_PROFILING_EXCLUDE_PATIENTS,
    apply_genomic_profiling_exclusions,
    export_for_r,
    is_nested_genomic_profiling,
)

OUT_SVG = FIG_DIR / "figS1A_oncoprint.svg"
OUT_PNG = FIG_DIR / "figS1A_oncoprint.png"
WORK_DIR = FIG_DIR / "_figS1A_r_work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
RSCRIPT = shutil.which("Rscript")
if RSCRIPT is None:
    raise RuntimeError(
        "Rscript not found on PATH. Install R and the packages listed in README.md (Fig S1A)."
    )

from IPython.display import display
from dlbcl.dlbcl_io import log_wrote, log_saved, rel_path, write_supplementary_table
subset = None  # nb8 classifier associations: filtered AnnData or None for all patients


AnnData: /Users/troynoordenbos/code/public_repo/DLBCL_location_2026/data/DLBCL_location_2026.h5ad
Figures: /Users/troynoordenbos/code/public_repo/DLBCL_location_2026/figures/supplemental_fig1


## Fig S1A — Genomic oncoprint
Unified oncoprint of pathogenic variants and copy-number alterations, columns ordered by anatomical location (PCNS, bone, nodal, testis).
**Data:** `data/DLBCL_location_2026.h5ad` only — tables under `adata.uns['genomic_profiling']`:
| Key | Description |
|-----|-------------|
| `events` | `patient_id`, `gene`, `alteration` (NON, STOP, HOT, GAIN, …) |
| `location_class` | `patient_id`, `Location`, `Location_class` for column annotation |
**Rendering:** the R code below is embedded in this notebook and executed via `Rscript` (requires R with **ComplexHeatmap** and **svglite** — see `README.md`).

## Load `adata.uns['genomic_profiling']`

In [2]:
adata = sc.read_h5ad(ADATA_PATH)
gp = apply_genomic_profiling_exclusions(adata.uns.get("genomic_profiling", {}))

if not is_nested_genomic_profiling(gp):
    raise RuntimeError(
        "Missing nested adata.uns['genomic_profiling']. "
        "Download an up-to-date DLBCL_location_2026.h5ad from Zenodo (see README.md)."
    )

events = pd.DataFrame(gp["events"])
location_class = pd.DataFrame(gp["location_class"])

print(f"Excluded patients: {sorted(GENOMIC_PROFILING_EXCLUDE_PATIENTS)}")
print(
    f"Events: {len(events):,} | genes: {events['gene'].nunique()} | "
    f"patients: {events['patient_id'].nunique()}"
)
display(location_class["Location_class"].value_counts().to_frame("n"))

r_paths = export_for_r(gp, WORK_DIR)


Excluded patients: ['T11', 'T22', 'T35', 'T47', 'T5', 'T57']
Events: 667 | genes: 126 | patients: 61


,n
Location_class,
Bone,20
PCNS,19
Nodal,16
Testis,9


## Render Fig S1A (embedded R)
The oncoprint renderer is defined in the next cell and written to a temporary `.R` file at run time.

In [3]:
ONCOPRINT_R_SCRIPT = '''# Unified oncoprint — reads pre-built event tables (from adata.uns['genomic_profiling']).

suppressPackageStartupMessages({
  library(ComplexHeatmap)
  library(grid)
  library(svglite)
})

args <- commandArgs(trailingOnly = TRUE)
get_arg <- function(flag, default = NULL) {
  i <- match(flag, args)
  if (is.na(i) || i >= length(args)) return(default)
  args[[i + 1]]
}

events_file <- get_arg("--events", "oncoprint_events.csv")
location_file <- get_arg("--location", "oncoprint_location_class.csv")
output_svg <- get_arg("--output", "Oncoprint_Unified_PCNS_Bone_Nodal_Testis.svg")
plot_family <- get_arg("--font", "DejaVu Sans")

location_order <- c("PCNS", "Bone", "Nodal", "Testis")

Onco_Cols <- c(
  "STAR" = "#6a78ff",
  "NON" = "#63d16b",
  "SPLI" = "#d1a252",
  "STOP" = "#c34600",
  "HOT" = "#7d6c3c",
  "FRAM" = "#7ca9b6",
  "REAR" = "#fff2b8",
  "GAIN" = "#001b21",
  "LOSS" = "#ffa19f"
)

Location_Cols <- c(
  "PCNS" = "#7F3C8D",
  "Bone" = "#11A579",
  "Nodal" = "#3969AC",
  "Testis" = "#F2B701"
)

alter_fun <- list(
  background = function(x, y, w, h) {
    grid.rect(x, y, w * 0.9, h * 0.9, gp = gpar(fill = "lightgray", col = NA))
  },
  NON = function(x, y, w, h) {
    grid.rect(x, y, w * 0.9, h * 0.9, gp = gpar(fill = Onco_Cols["NON"], col = NA))
  },
  STOP = function(x, y, w, h) {
    grid.rect(x, y, w * 0.8, h * 0.9, gp = gpar(fill = Onco_Cols["STOP"], col = NA))
  },
  SPLI = function(x, y, w, h) {
    grid.rect(x, y, w * 0.9, h * 0.6, gp = gpar(fill = Onco_Cols["SPLI"], col = NA))
  },
  STAR = function(x, y, w, h) {
    grid.rect(x, y, w * 0.9, h * 0.4, gp = gpar(fill = Onco_Cols["STAR"], col = NA))
  },
  HOT = function(x, y, w, h) {
    grid.rect(x, y, w * 0.9, h * 0.2, gp = gpar(fill = Onco_Cols["HOT"], col = NA))
  },
  FRAM = function(x, y, w, h) {
    grid.rect(x, y, w * 0.6, h * 0.9, gp = gpar(fill = Onco_Cols["FRAM"], col = NA))
  },
  REAR = function(x, y, w, h) {
    grid.rect(x, y, w * 0.4, h * 0.9, gp = gpar(fill = Onco_Cols["REAR"], col = NA))
  },
  GAIN = function(x, y, w, h) {
    grid.rect(x, y, w * 0.2, h * 0.9, gp = gpar(fill = Onco_Cols["GAIN"], col = NA))
  },
  LOSS = function(x, y, w, h) {
    grid.rect(x, y, w * 0.2, h * 0.9, gp = gpar(fill = Onco_Cols["LOSS"], col = NA))
  }
)

Onco_legend <- list(
  title = "Alterations",
  at = c("NON", "STOP", "SPLI", "STAR", "HOT", "FRAM", "REAR", "GAIN", "LOSS"),
  labels = c(
    "Nonsynonymous",
    "Stopgain/stoploss",
    "Splice-site",
    "Startloss",
    "Hotspot",
    "Frameshift/nonframeshift indel",
    "Rearrangement",
    "Gain",
    "Loss"
  ),
  legend_gp = gpar(fill = unname(Onco_Cols[c("NON", "STOP", "SPLI", "STAR", "HOT", "FRAM", "REAR", "GAIN", "LOSS")])),
  title_gp = gpar(fontfamily = plot_family, fontsize = 10),
  labels_gp = gpar(fontfamily = plot_family, fontsize = 9)
)

events <- read.csv(events_file, stringsAsFactors = FALSE)
metadata <- read.csv(location_file, stringsAsFactors = FALSE)
metadata$Location_class <- factor(metadata$Location_class, levels = location_order)
metadata <- metadata[!is.na(metadata$Location_class), ]
metadata <- metadata[!duplicated(metadata$patient_id), ]

events <- events[events$patient_id %in% metadata$patient_id, c("patient_id", "gene", "alteration")]
events <- events[!duplicated(events), ]

gene_tab <- sort(table(events$gene), decreasing = TRUE)
genes <- names(gene_tab)

metadata <- metadata[order(metadata$Location_class, metadata$patient_id), ]
patients <- metadata$patient_id

onco_matrix <- matrix("", nrow = length(genes), ncol = length(patients), dimnames = list(genes, patients))

event_matrix <- aggregate(
  alteration ~ gene + patient_id,
  data = events,
  FUN = function(x) paste(sort(unique(x)), collapse = ";")
)

onco_matrix[cbind(event_matrix$gene, event_matrix$patient_id)] <- event_matrix$alteration

sort_variant_columns <- function(mat) {
  if (ncol(mat) <= 1) return(colnames(mat))
  binary <- mat != ""
  row_order <- order(rowSums(binary), decreasing = TRUE)
  binary <- binary[row_order, , drop = FALSE]
  order_data <- as.data.frame(t(-1L * binary), check.names = FALSE)
  order_data$total_alterations <- -colSums(binary)
  order_data$patient_id <- rownames(order_data)
  colnames(mat)[do.call(order, order_data)]
}

patient_order <- unlist(lapply(location_order, function(location_class) {
  class_patients <- metadata$patient_id[metadata$Location_class == location_class]
  class_patients <- intersect(class_patients, colnames(onco_matrix))
  sort_variant_columns(onco_matrix[, class_patients, drop = FALSE])
}), use.names = FALSE)

onco_matrix <- onco_matrix[, patient_order, drop = FALSE]

column_annotation_data <- metadata[metadata$patient_id %in% patient_order, c("patient_id", "Location_class")]
column_annotation_data <- column_annotation_data[order(match(column_annotation_data$patient_id, patient_order)), ]

location_annotation <- HeatmapAnnotation(
  Location = column_annotation_data$Location_class,
  col = list(Location = Location_Cols),
  annotation_name_gp = gpar(fontfamily = plot_family, fontsize = 9),
  annotation_legend_param = list(
    title_gp = gpar(fontfamily = plot_family, fontsize = 10),
    labels_gp = gpar(fontfamily = plot_family, fontsize = 9)
  ),
  simple_anno_size = unit(0.35, "cm")
)

svg_width <- max(14, ncol(onco_matrix) * 0.18 + 4)
svg_height <- max(10, sum(rowSums(onco_matrix != "") > 0) * 0.16 + 3)

svglite(output_svg, width = svg_width, height = svg_height)
oncoprint_plot <- oncoPrint(
  onco_matrix,
  alter_fun = alter_fun,
  col = Onco_Cols,
  remove_empty_rows = TRUE,
  remove_empty_columns = FALSE,
  show_column_names = TRUE,
  column_names_gp = gpar(fontsize = 6, fontfamily = plot_family),
  column_title_gp = gpar(fontsize = 12, fontfamily = plot_family),
  alter_fun_is_vectorized = TRUE,
  heatmap_legend_param = Onco_legend,
  row_names_gp = gpar(fontsize = 8, fontface = "italic", fontfamily = plot_family),
  row_names_side = "left",
  pct_side = "right",
  top_annotation = location_annotation,
  column_split = column_annotation_data$Location_class,
  cluster_column_slices = FALSE,
  column_gap = unit(2, "mm")
)
draw(
  oncoprint_plot,
  heatmap_legend_side = "right",
  annotation_legend_side = "right",
  merge_legends = TRUE
)
dev.off()

cat("Wrote", output_svg, "\n")
'''

r_file = WORK_DIR / "render_oncoprint.R"
r_file.write_text(ONCOPRINT_R_SCRIPT)

cmd = [
    RSCRIPT,
    str(r_file),
    "--events", str(r_paths["events"]),
    "--location", str(r_paths["location"]),
    "--output", str(OUT_SVG),
]
print(" ".join(rel_path(Path(a), REPO_ROOT) if not str(a).startswith("-") else str(a) for a in cmd))
result = subprocess.run(cmd, cwd=REPO_ROOT, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError(f"Rscript failed (exit {result.returncode})")

if shutil.which("rsvg-convert"):
    subprocess.run(
        ["rsvg-convert", "-d", "300", "-o", str(OUT_PNG), str(OUT_SVG)],
        check=True,
    )
    log_wrote(OUT_PNG, REPO_ROOT)

log_wrote(OUT_SVG, REPO_ROOT)


Rscript figures/supplemental_fig1/_figS1A_r_work/render_oncoprint.R --events figures/supplemental_fig1/_figS1A_r_work/oncoprint_events.csv --location figures/supplemental_fig1/_figS1A_r_work/oncoprint_location_class.csv --output figures/supplemental_fig1/figS1A_oncoprint.svg


null device 
          1 
Wrote /Users/troynoordenbos/code/public_repo/DLBCL_location_2026/figures/supplemental_fig1/figS1A_oncoprint.svg 

Wrote figures/supplemental_fig1/figS1A_oncoprint.svg


## Fig S1B — Genomic classifier composition (LymphPlex, HMRN)
**Fig S1B** is reproduced in **Fig 1** — see `notebooks/fig1.ipynb`, section **Fig 1D — genomic classifier composition** (row-normalized stacked bars for LymphPlex and HMRN). No code is duplicated here.